# Exercise 6

## Predict rating using LSTM


In [1]:
import pandas as pd

In [2]:
dataTraining = pd.read_csv('https://github.com/sergiomora03/AdvancedTopicsAnalytics/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)

In [3]:
plots = dataTraining['plot']
y = (dataTraining['rating'] >= dataTraining['rating'].mean()).astype(int)

In [4]:
plots

3107    most is the story of a single father who takes...
900     a serial killer decides to teach the secrets o...
6724    in sweden ,  a female blackmailer with a disfi...
4704    in a friday afternoon in new york ,  the presi...
2582    in los angeles ,  the editor of a publishing h...
                              ...                        
8417    " our marriage ,  their wedding .  "  it ' s l...
1592    the wandering barbarian ,  conan ,  alongside ...
1723    like a tale spun by scheherazade ,  kismet fol...
7605    mrs .  brisby ,  a widowed mouse ,  lives in a...
215     tinker bell journey far north of never land to...
Name: plot, Length: 7895, dtype: object

In [5]:
y

3107    1
900     0
6724    1
4704    1
2582    1
       ..
8417    0
1592    0
1723    0
7605    1
215     1
Name: rating, Length: 7895, dtype: int32

# Exercise 6.1

- Remove stopwords
- Lowercase
- split the text in words
- pad_sequences

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USUARIO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# --- Preprocesamiento de texto ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in text.split() if word not in stop_words]
    return ' '.join(words)

cleaned_texts = dataTraining['plot'].apply(clean_text)

# --- Tokenización y padding ---
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(cleaned_texts)
sequences = tokenizer.texts_to_sequences(cleaned_texts)
padded_sequences = pad_sequences(sequences, padding='post', maxlen=200)

X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y, test_size=0.2, random_state=42)

# --- Dataset personalizado ---
class MovieDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MovieDataset(X_train, y_train)
test_dataset = MovieDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


# Exercise 6.2

Create a SimpleRNN neural network to predict the rating of a movie

Calculate the testing set accuracy

In [15]:
# --- Modelo con SimpleRNN ---
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SimpleRNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        rnn_out, _ = self.rnn(x)
        out = rnn_out[:, -1, :]  # última salida de la secuencia
        out = self.dropout(out)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return self.sigmoid(out).squeeze()

In [16]:
# --- Entrenamiento ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleRNNClassifier(vocab_size=10000, embed_dim=64, hidden_dim=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 137.6853
Epoch 2, Loss: 137.3894
Epoch 3, Loss: 137.0329
Epoch 4, Loss: 137.0481
Epoch 5, Loss: 136.8787


In [17]:
# --- Evaluación ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        correct += (predicted == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total
print(f"\n✅ Test Accuracy (SimpleRNN): {accuracy:.4f}")



✅ Test Accuracy (SimpleRNN): 0.5465


En este experimento se desarrolló un modelo de red neuronal utilizando la arquitectura SimpleRNN con PyTorch para predecir si una película tiene una calificación superior al promedio, a partir de su sinopsis. El texto fue preprocesado eliminando signos de puntuación, palabras vacías (stopwords) y convirtiendo todo a minúsculas. Posteriormente, se aplicó tokenización conservando las 10,000 palabras más frecuentes y se normalizó la longitud de las secuencias a 200 tokens mediante padding.

La arquitectura del modelo incluye una capa de embedding que transforma los tokens en vectores densos de dimensión 64, seguida de una capa SimpleRNN con 64 unidades ocultas. A continuación, se agregan capas Dropout para evitar el sobreajuste y dos capas densas: una intermedia con 32 neuronas y función de activación ReLU, y una final con una única neurona con activación sigmoide para la clasificación binaria.

El modelo se entrenó durante cinco épocas utilizando el optimizador Adam y la función de pérdida BCELoss. Al evaluar el modelo sobre el conjunto de prueba, se obtuvo una precisión de 54.65%, un resultado comparable al de las arquitecturas más sofisticadas (LSTM y GRU), lo cual sugiere que el principal limitante no está en la complejidad del modelo, sino en la calidad y riqueza de la información contenida en las sinopsis.

# Exercise 6.3

Create a LSTM neural network to predict the rating of a movie

Calculate the testing set accuracy

In [18]:
# --- Modelo LSTM ---
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]  # Tomamos la última salida temporal
        out = self.dropout(out)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return self.sigmoid(out).squeeze()

# --- Inicialización y entrenamiento ---
model = LSTMClassifier(vocab_size=10000, embed_dim=64, hidden_dim=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [19]:
# --- Inicialización y entrenamiento ---
model = LSTMClassifier(vocab_size=10000, embed_dim=64, hidden_dim=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 137.2419
Epoch 2, Loss: 137.0977
Epoch 3, Loss: 137.0766
Epoch 4, Loss: 137.0209
Epoch 5, Loss: 136.7987


In [20]:
# --- Evaluación ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        correct += (predicted == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total
print(f"\n✅ Test Accuracy (LSTM): {accuracy:.4f}")


✅ Test Accuracy (LSTM): 0.5478


En este ejercicio se implementó un modelo de red neuronal basado en la arquitectura LSTM (Long Short-Term Memory) utilizando PyTorch. El objetivo fue clasificar si una película tiene una calificación superior al promedio, empleando como entrada su sinopsis textual. Para ello, se preprocesaron los textos eliminando signos de puntuación, palabras vacías (stopwords) y aplicando tokenización, limitando el vocabulario a las 10,000 palabras más frecuentes. Luego, se aplicó padding para igualar la longitud de las secuencias a 200 tokens.

La arquitectura del modelo incluye una capa de embedding que transforma los tokens en vectores numéricos de 64 dimensiones, seguida de una capa LSTM con 64 unidades ocultas que permite capturar dependencias secuenciales en el texto. Posteriormente, se incorpora una capa de Dropout para evitar el sobreajuste, una capa densa intermedia de 32 neuronas con activación ReLU y una capa de salida con activación sigmoide, adecuada para la clasificación binaria.

El modelo fue entrenado durante cinco épocas utilizando el optimizador Adam y la función de pérdida BCELoss. Al evaluarlo sobre el conjunto de prueba, se obtuvo una precisión de 54.78%, un resultado comparable al de modelos con arquitecturas similares como GRU o SimpleRNN, lo que sugiere que el límite principal está más en las características del texto que en la arquitectura del modelo.

# Exercise 6.4

Create a GRU neural network to predict the rating of a movie

Calculate the testing set accuracy

In [21]:
# --- Modelo GRU ---
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        gru_out, _ = self.gru(x)
        out = gru_out[:, -1, :]  # Última salida temporal
        out = self.dropout(out)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return self.sigmoid(out).squeeze()

In [22]:
# --- Inicialización y entrenamiento ---
model = GRUClassifier(vocab_size=10000, embed_dim=64, hidden_dim=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 137.2192
Epoch 2, Loss: 137.1315
Epoch 3, Loss: 137.0257
Epoch 4, Loss: 136.6413
Epoch 5, Loss: 131.6751


In [23]:
# --- Evaluación ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        correct += (predicted == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total
print(f"\n✅ Test Accuracy (GRU): {accuracy:.4f}")



✅ Test Accuracy (GRU): 0.5693


En este ejercicio se implementó un modelo de red neuronal basado en la arquitectura GRU (Gated Recurrent Unit) utilizando PyTorch. El objetivo fue predecir si la calificación de una película está por encima del promedio, a partir del texto de su sinopsis. Se mantuvo el mismo preprocesamiento de ejercicios anteriores: limpieza del texto, eliminación de stopwords, tokenización con un vocabulario limitado a 10,000 palabras, y padding de las secuencias hasta 200 tokens.

La arquitectura del modelo incluye una capa de embedding de 64 dimensiones, seguida por una capa GRU con 64 unidades ocultas que permite modelar dependencias en las secuencias de texto de forma más eficiente que una LSTM, al utilizar una estructura más simple con menos parámetros. A continuación, se aplican capas de Dropout para reducir el riesgo de sobreajuste, una capa densa intermedia con 32 neuronas y activación ReLU, y finalmente una capa de salida con activación sigmoide para realizar clasificación binaria.

El modelo se entrenó durante cinco épocas utilizando el optimizador Adam y la función de pérdida BCELoss. Al ser evaluado sobre el conjunto de prueba, alcanzó una precisión del 56.93%, superando a los modelos anteriores construidos con LSTM (54.78%) y SimpleRNN (54.65%). Esto demuestra que, para este conjunto de datos, la arquitectura GRU ofrece un mejor equilibrio entre capacidad de aprendizaje y eficiencia computacional.